In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.4 MB/s eta 0:00:00


In [ ]:
# Autoencoder Model for Anomaly Detection (PyTorch) - Optimized with Optuna

import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Paths
base_path = '/content/drive/MyDrive/Colab Notebooks/predictive maintenance dataset/Dataset/Wind Farm A'
output_path = os.path.join(base_path, 'output')

In [ ]:
# Ensure output directory exists
os.makedirs(output_path, exist_ok=True)

In [ ]:
# Load combined dataset
combined_df = pd.read_csv(os.path.join(output_path, 'combined_wind_farm_data.csv'))
combined_df

,asset_id,id,train_test,status_type_id,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,wind_speed_3_max,...,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg
0,0,0,train,0,22.0,302.9,129.4,1.7,1.7,11.7,...,-496.0,0.0,0.0,-1280.0,-496.0,0.0,0.0,0.0,0.0,26.0
1,0,1,train,0,22.0,307.1,133.6,1.7,1.7,8.3,...,-490.0,0.0,0.0,-1278.0,-490.0,0.0,0.0,0.0,0.0,25.0
2,0,2,train,0,22.0,340.6,167.1,0.9,0.9,5.9,...,-490.0,0.0,0.0,-1356.0,-490.0,0.0,0.0,0.0,0.0,25.0
3,0,3,train,0,22.0,124.4,-49.1,1.5,1.5,7.1,...,-509.0,0.0,0.0,-1274.0,-509.0,0.0,0.0,0.0,0.0,26.0
4,0,4,train,0,22.0,66.2,-107.3,1.0,1.0,2.1,...,-499.0,0.0,0.0,-1284.0,-499.0,0.0,0.0,0.0,0.0,26.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1196742,11,54062,prediction,5,15.0,117.8,4.8,2.7,2.7,6.2,...,-328.0,0.0,0.0,-772.0,-328.0,1.4,2.7,0.0,1.1,18.0
1196743,11,54063,prediction,5,16.0,69.7,-32.0,2.3,2.3,7.2,...,-231.0,0.0,0.0,-725.0,-231.0,0.3,2.1,0.0,0.7,18.0
1196744,11,54064,prediction,5,16.0,69.8,-16.9,2.8,2.8,5.6,...,-492.0,0.0,0.0,-844.0,-492.0,1.9,2.1,1.6,0.1,19.0
1196745,11,54065,prediction,5,16.0,63.5,-10.6,2.4,2.4,4.8,...,-228.0,0.0,0.0,-731.0,-228.0,0.4,1.8,0.0,0.7,19.0


In [ ]:
# Select sensor data for training
sensor_columns = [col for col in combined_df.columns if 'sensor' in col or 'wind_speed' in col or 'power' in col]
X = combined_df[sensor_columns].values
for i in range(X.shape[1]):
    nan_mask = np.isnan(X[:, i])
    X[nan_mask, i] = np.nanmean(X[:, i])
print("NaN values in dataset:", np.isnan(X).sum())
print("Inf values in dataset:", np.isinf(X).sum())

NaN values in dataset: 0
Inf values in dataset: 0


In [ ]:
# Filter normal operation data for training
normal_data = combined_df[combined_df['status_type_id'] == 0]
X = normal_data[sensor_columns].values


In [ ]:
# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Split data into training (75%) and validation (25%)
X_train, X_val = train_test_split(X_scaled, test_size=0.25, random_state=42)

In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
X_val_tensor = torch.FloatTensor(X_val)

In [ ]:
# Move tensors to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_train_tensor = X_train_tensor.to(device)
X_val_tensor = X_val_tensor.to(device)

In [ ]:
# Create DataLoader
batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor), batch_size=batch_size, shuffle=False)

In [ ]:
# Hyperparameter Optimization with Optuna
def objective(trial):
    hidden_layers = trial.suggest_categorical('hidden_layers', [(44, 25, 4, 25, 44)])
    learning_rate = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    noise_level = trial.suggest_float('noise', 0.05, 0.1)

    class Autoencoder(nn.Module):
        def __init__(self, input_dim, hidden_layers):
            super(Autoencoder, self).__init__()
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden_layers[0]),
                nn.ReLU(),
                nn.Linear(hidden_layers[0], hidden_layers[1]),
                nn.ReLU(),
                nn.Linear(hidden_layers[1], hidden_layers[2])
            )
            self.decoder = nn.Sequential(
                nn.Linear(hidden_layers[2], hidden_layers[3]),
                nn.ReLU(),
                nn.Linear(hidden_layers[3], hidden_layers[4]),
                nn.ReLU(),
                nn.Linear(hidden_layers[4], input_dim)
            )

        def forward(self, x):
            encoded = self.encoder(x)
            decoded = self.decoder(encoded)
            return decoded

    model = Autoencoder(X_train.shape[1], hidden_layers).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training the Autoencoder
    num_epochs = 200
    best_loss = float('inf')
    early_stop_counter = 0
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            data = batch[0].to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, data)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        # Validation loss
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                data = batch[0].to(device)
                outputs = model(data)
                loss = criterion(outputs, data)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        if val_loss < best_loss:
            best_loss = val_loss
            early_stop_counter = 0
        else:
            early_stop_counter += 1

        if early_stop_counter >= 3:
            break

    return best_loss

In [ ]:
# Run Optuna Optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)


[I 2025-03-17 12:16:35,380] A new study created in memory with name: no-name-e0a245e7-1982-4a9e-9afe-c2798791593f
/usr/local/lib/python3.11/dist-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (44, 25, 4, 25, 44) which is of type tuple.
  warnings.warn(message)
[I 2025-03-17 12:18:05,123] Trial 0 finished with value: inf and parameters: {'hidden_layers': (44, 25, 4, 25, 44), 'lr': 0.004198951778484099, 'noise': 0.08095127114734016}. Best is trial 0 with value: inf.
/usr/local/lib/python3.11/dist-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (44, 25, 4, 25, 44) which is of type tuple.
  warnings.warn(message)
[I 2025-03-17 12:19:29,421] Trial 1 finished with value: inf and parameters: {'hidden_layers': (44, 25, 4, 25, 44), 'lr': 

In [ ]:
# Best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

Best Hyperparameters: {'hidden_layers': (44, 25, 4, 25, 44), 'lr': 0.004198951778484099, 'noise': 0.08095127114734016}
